In [3]:
'''
S&P 500 (^GSPC, adjusted close) drawdowns deeper than THRESHOLD.

start   last all-time high before the drawdown (peak)
trough  lowest close of the drawdown
end     first close back at or above the peak (NaT = not recovered yet)
'''

import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / '_metrics'))
from portfolio import load_prices

THRESHOLD = 0.15

px = load_prices()['^GSPC'].dropna()
peak = px.cummax()
dd = px / peak - 1

# a new episode starts on every close at the running peak
episode = (px >= peak).cumsum()

rows = []
for _, d in dd.groupby(episode):
    if d.min() > -THRESHOLD:
        continue
    start, trough = d.index[0], d.idxmin()
    after = px.index[px.index > d.index[-1]]
    end = after[0] if len(after) else pd.NaT
    rows.append({
        'start': start.date(),
        'trough': trough.date(),
        'end': end.date() if pd.notna(end) else pd.NaT,
        'drawdown': d.min(),
        'days_to_trough': len(px.loc[start:trough]) - 1,
        'days_to_end': len(px.loc[start:end]) - 1 if pd.notna(end) else None,
    })

drawdowns = pd.DataFrame(rows)
print(f'^GSPC {px.index[0].date()}..{px.index[-1].date()}: '
      f'{len(drawdowns)} drawdowns deeper than {THRESHOLD:.0%}')
drawdowns.style.format({'drawdown': '{:.1%}'})

^GSPC 1988-01-04..2025-12-30: 8 drawdowns deeper than 15%


,start,trough,end,drawdown,days_to_trough,days_to_end
0,1990-07-16,1990-10-11,1991-02-13,-19.9%,62,148
1,1998-07-17,1998-08-31,1998-11-23,-19.3%,31,90
2,2000-03-24,2002-10-09,2007-05-30,-49.1%,637,1803
3,2007-10-09,2009-03-09,2013-03-28,-56.8%,355,1376
4,2018-09-20,2018-12-24,2019-04-23,-19.8%,65,146
5,2020-02-19,2020-03-23,2020-08-18,-33.9%,23,126
6,2022-01-03,2022-10-12,2024-01-19,-25.4%,195,513
7,2025-02-19,2025-04-08,2025-06-27,-18.9%,34,89
